## Task 3: Soft Label Loss

**Goal:** Use Label Smoothing BCE as an alternative to cross-entropy on the same imbalanced dataset.

1. Implement `LabelSmoothingBinaryCrossEntropy`:

```python
class LabelSmoothingBinaryCrossEntropy(nn.Module):
    """
    Label Smoothing Binary Cross-Entropy.
    Replaces hard binary targets {0, 1} with smoothed targets:
      y_smoothed = y * (1 - eps) + (1 - y) * eps
    Prevents overconfidence and improves calibration.
    Works as a regulariser for noisy labels.
    Args:
        eps (float): smoothing strength. Typical range: 0.05 – 0.15
        reduction (str): 'mean' | 'sum' | 'none'
    """
    def __init__(self, eps: float = 0.1, reduction: str = "mean"):
        super().__init__()
        self.eps       = eps
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets          = targets.float()
        smoothed_targets = targets * (1.0 - self.eps) + (1.0 - targets) * self.eps
        return F.binary_cross_entropy_with_logits(logits, smoothed_targets,
                                                   reduction=self.reduction)
```

2. Implement, train, evaluate and compare analogously to Task 2 — for several values of the smoothing coefficient `eps`.

**Assignment:** Implement and train a classifier on a strongly imbalanced dataset — compare classification quality for cross-entropy and Soft Label Loss.

In [5]:
IMBALANCE_RATIO = 0.90

BATCH_SIZE   = 128
EPOCHS       = 1000
LR           = 1e-3
WEIGHT_DECAY = 1e-4

In [6]:
from utils import make_imbalanced_dataset, BinaryClassifierMLP, make_loaders
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = BinaryClassifierMLP().to(device)

X, y = make_imbalanced_dataset(imbalance_ratio=IMBALANCE_RATIO)
print(f"Class 0: {(y == 0).sum():.0f}  Class 1: {(y == 1).sum():.0f}")

train_loader, val_loader, test_loader = make_loaders(X, y, batch_size=BATCH_SIZE, weighted_sampler=True, squeeze_y=True)

Class 0: 10696  Class 1: 1304


In [7]:
from utils import train_baseline
from utils import LabelSmoothingBinaryCrossEntropy
import torch.nn as nn

print("Training with BCE: ")
model_bce = train_baseline(model, train_loader, val_loader,
                           criterion=nn.BCEWithLogitsLoss(), device=device,
                           weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR
                           )

print("\nTraining with Label Smoothing BCE: ")
model_lsbce = train_baseline(model, train_loader, val_loader,
                             criterion=LabelSmoothingBinaryCrossEntropy(),
                             device=device, weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR
                             )

Training with BCE: 
Epoch 100/1000  train=0.0708  val=0.1803
Epoch 200/1000  train=0.0541  val=0.2088
Epoch 300/1000  train=0.0405  val=0.2231
Epoch 400/1000  train=0.0309  val=0.2359
Epoch 500/1000  train=0.0250  val=0.2477
Epoch 600/1000  train=0.0222  val=0.2531
Epoch 700/1000  train=0.0203  val=0.2590
Epoch 800/1000  train=0.0228  val=0.2577
Epoch 900/1000  train=0.0221  val=0.2604
Epoch 1000/1000  train=0.0221  val=0.2521

Training with Label Smoothing BCE: 
Epoch 100/1000  train=0.3773  val=0.3922
Epoch 200/1000  train=0.3657  val=0.3928
Epoch 300/1000  train=0.3642  val=0.3919
Epoch 400/1000  train=0.3597  val=0.3860
Epoch 500/1000  train=0.3572  val=0.3833
Epoch 600/1000  train=0.3558  val=0.3861
Epoch 700/1000  train=0.3531  val=0.3812
Epoch 800/1000  train=0.3536  val=0.3841
Epoch 900/1000  train=0.3522  val=0.3790
Epoch 1000/1000  train=0.3554  val=0.3786


In [8]:
from utils import get_probs, compute_clf_metrics

y_true, probs_bce = get_probs(model_bce, test_loader, device=device)
y_true, probs_lsbce = get_probs(model_lsbce, test_loader, device=device)

m_bce = compute_clf_metrics(y_true, probs_bce)
model_lsbce = compute_clf_metrics(y_true, probs_lsbce)

print(f"\n{'Metric':<12} {'BCE':>10} {'Label Smoothing BCE Loss':>12}")
print("=" * 36)
for key in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
    print(f"{key:<12} {m_bce[key]:>10.4f} {model_lsbce[key]:>12.4f}")
print("=" * 36)
print(f"\nBCE        — TP={m_bce['tp']}  FP={m_bce['fp']}  TN={m_bce['tn']}  FN={m_bce['fn']}")
print(f"Label Smoothing BCE Loss — TP={model_lsbce['tp']}  FP={model_lsbce['fp']}  TN={model_lsbce['tn']}  FN={model_lsbce['fn']}")


Metric              BCE Label Smoothing BCE Loss
accuracy         0.9708       0.9708
precision        0.8684       0.8684
recall           0.8319       0.8319
f1               0.8498       0.8498
roc_auc          0.9562       0.9562
pr_auc           0.8944       0.8944

BCE        — TP=198  FP=30  TN=2132  FN=40
Label Smoothing BCE Loss — TP=198  FP=30  TN=2132  FN=40


BCE (Binary Cross-Entropy) is a standard binary classification loss function.

Soft Label Loss (Label Smoothing) is BCE loss function that doesn't use {0, 1} target classes and instead {eps, 1-eps}. It doesn't change gradient changes but prevents model from becoming overconfident, it works similarly to regularization and probability calibration.

BCE — balanced or slightly imbalanced data
Soft Label — labels may be noisy, model is overconfident